# 07 — Train the FNO

Self-contained after data acquisition: rebuilds the chronological splits and
loaders, trains the FNO, and saves everything needed to reconstruct the run —
configuration, environment, epoch history, and the best-validation checkpoint.

The training loop stays here rather than moving into a framework. Only the
bookkeeping that notebooks `05`, `07`, and `10` all repeat lives in
`oisst_fno.experiment`.

## What reproducible means here

Seeding fixes Python, NumPy, and PyTorch. It does **not** guarantee
bitwise-identical results on GPU. The remaining sources of variation are printed
with every run, so a run that cannot be reproduced exactly can still be
attributed to a specific environment.

In [ ]:
from pathlib import Path
import time

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from tqdm.auto import trange

from oisst_fno.data import ForecastSpec, SSTWindowDataset, Standardizer, open_oisst, temporal_split
from oisst_fno.experiment import (
    GPU_NONDETERMINISM_NOTES,
    EpochRecord,
    ExperimentConfig,
    TrainingHistory,
    amp_is_supported,
    collect_environment,
    diagnose_learning_curves,
    gradient_norm,
    set_global_seed,
)
from oisst_fno.metrics import masked_mse_loss, parameter_count
from oisst_fno.model import FNO2d

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = sorted((ROOT / "data" / "raw").glob("oisst_*_ne_atlantic.nc"))[-1]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
DETERMINISTIC = False  # set True to pin cuDNN algorithms; slower, and may raise on GPU
set_global_seed(SEED, deterministic=DETERMINISTIC)

environment = collect_environment()
print("device:", DEVICE)
print("git commit:", environment.git_commit, "(dirty)" if environment.git_dirty else "")
print("torch:", environment.torch_version, "| cuda:", environment.cuda_version)

if DEVICE.type == "cuda":
    print("\nRemaining sources of run-to-run variation on GPU:")
    for note in GPU_NONDETERMINISM_NOTES:
        print(f"  - {note}")

## Splits and loaders

The standardizer is fitted on the **training split only**, before any window is
built, so no validation or test information reaches preprocessing.

In [ ]:
TRAIN_END = "2024-12-31"
VALIDATION_END = "2025-12-31"
SPEC = ForecastSpec(lookback_days=14, horizon_days=7)
BATCH_SIZE = 16

sst = open_oisst(DATA_PATH)["sst"]
train_da, val_da, _ = temporal_split(sst, TRAIN_END, VALIDATION_END)

scaler = Standardizer.fit(train_da.values)  # training split only
train_ds = SSTWindowDataset(scaler.transform(train_da.values), SPEC)
val_ds = SSTWindowDataset(scaler.transform(val_da.values), SPEC)


def collate_with_mask(batch):
    xs, ys, masks = zip(*batch)
    x = torch.stack(xs)
    y = torch.stack(ys)
    mask = torch.stack(masks)
    return torch.cat((x, mask), dim=1), y, mask


# A seeded generator keeps shuffling reproducible across runs.
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_with_mask,
    num_workers=0,
    generator=loader_generator,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_with_mask,
    num_workers=0,
)
print(f"train windows: {len(train_ds)} | validation windows: {len(val_ds)}")

In [ ]:
MODEL_CONFIG = {
    "in_channels": SPEC.lookback_days + 1,
    "out_channels": 1,
    "width": 48,
    "modes_y": 16,
    "modes_x": 16,
    "depth": 4,
    "padding": 8,
}
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 50
PATIENCE = 8
CLIP_NORM = 1.0
USE_AMP = amp_is_supported(DEVICE)

model = FNO2d(**MODEL_CONFIG).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
grad_scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

print(f"trainable parameters: {parameter_count(model):,}")
print(f"mixed precision: {USE_AMP}")

## Training loop

Per epoch it records train and validation loss, learning rate, wall-clock
seconds, gradient norm, and peak GPU memory.

A non-finite loss aborts immediately rather than poisoning the checkpoint. An FNO
with too high a learning rate can diverge in a single step, and a silently saved
NaN model would later look like a legitimate negative result.

Mixed precision is used only on CUDA; the CPU path runs in full precision so
results stay correct on a laptop.

In [ ]:
def run_epoch(model, loader, *, optimizer=None):
    """Run one pass. Returns (mean loss, gradient norm); the norm is None when evaluating."""
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_examples = 0
    last_grad_norm = None

    for x, y, mask in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        mask = mask.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                prediction = model(x)
                loss = masked_mse_loss(prediction, y, mask)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss ({loss.item()}). Training stopped so the checkpoint "
                    "is not overwritten with a diverged model. Lower the learning rate "
                    "or tighten gradient clipping."
                )

            if training:
                grad_scaler.scale(loss).backward()
                # Unscale before clipping and before reading the norm, so both are
                # measured in real units rather than AMP-scaled ones.
                grad_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CLIP_NORM)
                last_grad_norm = gradient_norm(model.parameters())
                grad_scaler.step(optimizer)
                grad_scaler.update()

        batch_size = x.shape[0]
        total_loss += float(loss.detach()) * batch_size
        total_examples += batch_size

    return total_loss / max(total_examples, 1), last_grad_norm

In [ ]:
BEST_PATH = ROOT / "artifacts" / "models" / "fno_best.pt"
HISTORY_PATH = ROOT / "artifacts" / "metrics" / "fno_training_history.json"
CONFIG_PATH = ROOT / "artifacts" / "metrics" / "fno_experiment.json"
BEST_PATH.parent.mkdir(parents=True, exist_ok=True)

history = TrainingHistory()
best_val = float("inf")
stale_epochs = 0
stopped_early = False

for epoch in trange(EPOCHS):
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()

    train_loss, grad_norm_value = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss, _ = run_epoch(model, val_loader)

    current_lr = float(optimizer.param_groups[0]["lr"])
    scheduler.step()

    history.append(
        EpochRecord(
            epoch=epoch + 1,
            train_loss=train_loss,
            val_loss=val_loss,
            learning_rate=current_lr,
            seconds=time.perf_counter() - started,
            grad_norm=grad_norm_value,
            peak_gpu_mb=(
                torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == "cuda" else None
            ),
        )
    )

    # Only the best validation checkpoint is kept.
    if val_loss < best_val:
        best_val = val_loss
        stale_epochs = 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            stopped_early = True
            break

history.save(HISTORY_PATH)
print(f"best epoch {history.best.epoch} at val loss {history.best.val_loss:.5f}")
print(f"wall clock: {history.total_seconds:.1f}s over {len(history)} epochs")
if history.peak_gpu_mb is not None:
    print(f"peak GPU memory: {history.peak_gpu_mb:.0f} MiB")

## Saved configuration

Everything needed to set up a second run, written next to the checkpoint. A rerun
is configured from this file rather than from whatever the notebook happened to
contain at the time.

In [ ]:
config = ExperimentConfig(
    name="fno-baseline",
    seed=SEED,
    deterministic=DETERMINISTIC,
    data_path=str(DATA_PATH.relative_to(ROOT)),
    train_end=TRAIN_END,
    validation_end=VALIDATION_END,
    region={
        "lat_min": float(sst["lat"].min()),
        "lat_max": float(sst["lat"].max()),
        "lon_min": float(sst["lon"].min()),
        "lon_max": float(sst["lon"].max()),
    },
    lookback_days=SPEC.lookback_days,
    horizon_days=SPEC.horizon_days,
    scaler={"mean": scaler.mean, "std": scaler.std},
    model=MODEL_CONFIG,
    optimizer="AdamW",
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    epochs_requested=EPOCHS,
    early_stopping={
        "monitor": "val_loss",
        "mode": "min",
        "patience": PATIENCE,
        "stopped_early": stopped_early,
        "epochs_run": len(history),
        "best_epoch": history.best.epoch,
    },
    scheduler="CosineAnnealingLR",
    mixed_precision=USE_AMP,
    gradient_clip_norm=CLIP_NORM,
    notes="Standardizer fitted on the training split only.",
)
config.save(CONFIG_PATH, environment=environment)
print(CONFIG_PATH.read_text(encoding="utf-8")[:900])

## Learning curves and diagnosis

The verdict below is a blunt heuristic meant to prompt a look at the curves, not
to replace one. Read it together with the plot.

In [ ]:
import matplotlib.pyplot as plt

diagnosis = diagnose_learning_curves(history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs = [record.epoch for record in history.records]

axes[0].plot(epochs, history.train_losses, label="train")
axes[0].plot(epochs, history.val_losses, label="validation")
axes[0].axvline(history.best.epoch, color="grey", linestyle="--", label="best (saved)")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("masked standardized MSE")
axes[0].set_title(f"Learning curves - {diagnosis.verdict}")
axes[0].legend()

grad_norms = [r.grad_norm for r in history.records if r.grad_norm is not None]
if grad_norms:
    axes[1].plot(epochs[: len(grad_norms)], grad_norms)
    axes[1].set_yscale("log")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("gradient L2 norm (after clipping)")
    axes[1].set_title("Gradient norm")

plt.tight_layout()
plt.show()

print(diagnosis)

### Reading the verdict

- **underfit** — the run stopped while validation loss was still falling. Train
  longer or add capacity before drawing any conclusion about FNO skill; a model
  that was simply undertrained is not evidence against the method.
- **overfit** — validation loss rose after its minimum. The saved checkpoint is
  the best epoch, not the last, so the artifact is still usable, but the gap
  indicates more capacity than the data supports.
- **unstable** — the optimizer diverged or spiked. Lower the learning rate or clip
  harder. Results from an unstable run should not be reported.
- **converged** — proceed to evaluation in notebook `08`.

Whatever the verdict, a lower validation loss is **not** yet evidence of forecast
skill. That comparison happens against persistence and the other baselines on the
untouched test period, in notebooks `08` and `09`.